In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')

import warnings
warnings.filterwarnings('ignore')

# Dataset

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/eduardofc/data/refs/heads/main/airbnb_2019_2.csv")
df.head()

In [ ]:
# Eliminamos valores que no están bien en price (descubierto en el EDA)

df = df[df.price >= 10]  # las habs deben valer más de 10€

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

# EDA

## target

In [ ]:
# price (planteamiento)

X = df['price']    # <- 1) target cruda
# X = np.log(X + 1) # <- 2) target manipulada
# np.quantile(X, [.90, .95, .99]) <- 3) ensemble de modelos 
# 4) hacer la target multiclase (problema de clasificación)

plt.figure(figsize=(10,2))
sns.histplot(X)
plt.show()

# Manipulación de la fecha

In [ ]:
# last_review (opciones: tomar el año, tomar el mes, tomar meses desde la ultima review, tomar dias desde la ultima review)

df1 = df.copy()
df1['last_review'] = pd.to_datetime(df1['last_review'])

ref_date = pd.Timestamp('2019-12-31')
df1['last_review'] = (ref_date - df1['last_review']).dt.days

# Exploración unidimensional

In [ ]:
num_cols = ['latitude','longitude','minimum_nights','number_of_reviews','reviews_per_month', 'calculated_host_listings_count','availability_365','last_review']
cat_cols = ['neighbourhood_group','neighbourhood','room_type']

# ojo que los nulos no los plotea

In [ ]:
# numéricas

for col in num_cols:
    plt.figure(figsize=(10,2))
    plt.subplot(1,2,1)
    sns.histplot(data=df1, x=col)
    plt.subplot(1,2,2)
    sns.boxplot(data=df1, x=col)
    plt.show()

In [ ]:
# categóricas

for col in cat_cols:
    plt.figure(figsize=(10,2))
    sns.countplot(data=df1, x=col)
    plt.show()

## Exploración bidimensional

In [ ]:
# bidimensional (categórica vs numérica)

X = df1['price']
X = np.log(X)
X = df1[df1.price < 700]['price'] 

for col in cat_cols:
    plt.figure(figsize=(10,2))
    sns.boxplot(data=df1, y=X, x=col)
    plt.show()

In [ ]:
# bidimensional (dos categóricas)

plt.figure(figsize=(10,2))
sns.countplot(data=df1, x="neighbourhood_group", hue="room_type")
plt.show()

In [ ]:
# bidimensional (dos numéricas)

# X1 = np.log(df1["minimum_nights"] + 1)
# X2 = np.log(df1["number_of_reviews"] + 1)
X1 = df1['longitude']
X2 = df1['latitude']
X3 = df1["neighbourhood_group"]

sns.jointplot(x=X1, y=X2, hue=X3)  # ojo que los nulos no los plotea

In [ ]:
df_na_neigh = df1.copy()
df_na_neigh = df_na_neigh[df_na_neigh.neighbourhood_group.isna()]

X1 = df_na_neigh['longitude']
X2 = df_na_neigh['latitude']
X3 = df_na_neigh["neighbourhood_group"]

sns.jointplot(x=X1, y=X2, hue=X3)  # ojo que los nulos no los plotea

In [ ]:
sns.heatmap(df1.corr(numeric_only=True), annot=True)
plt.show()

# Nulos

In [ ]:
df2 = df1.copy()

In [ ]:
df2.isna().sum()

In [ ]:
X = df['room_type']
print(X.shape)
X = df[['room_type']]
print(X.shape)
X = df[['room_type', "availability_365"]]
print(X.shape)

In [ ]:
# simple

from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='most_frequent')

df2[['room_type']] = imputer.fit_transform(df2[['room_type']])

# hablar del fit, transform, fit-transform

In [ ]:
plt.figure(figsize=(10,2))
plt.subplot(1,2,1)
sns.countplot(data=df1, x='room_type')
plt.subplot(1,2,2)
sns.countplot(data=df2, x='room_type')
plt.show()

In [ ]:
imputer = SimpleImputer(strategy='median')
print(df2['reviews_per_month'].median())

df2[['reviews_per_month']] = imputer.fit_transform(df2[['reviews_per_month']])

In [ ]:
X1 = df1['reviews_per_month']
X2 = df2['reviews_per_month']

plt.figure(figsize=(10,2))
plt.subplot(1,2,1)
sns.histplot(X1, bins=10)
plt.subplot(1,2,2)
sns.histplot(X2, bins=10)
plt.show()

In [ ]:
X1 = np.log(df1['reviews_per_month'] + 1)
X2 = np.log(df2['reviews_per_month'] + 1)

plt.figure(figsize=(10,2))
plt.subplot(1,2,1)
sns.histplot(X1, bins=10)
plt.subplot(1,2,2)
sns.histplot(X2, bins=10)
plt.show()

In [ ]:
# knn

from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=5, weights='distance')

cols = ['latitude', 'longitude', 'price', 'calculated_host_listings_count', 'last_review']

df2[cols] = imputer.fit_transform(df2[cols])

In [ ]:
X1 = np.log(df1['last_review'] + 1)
X2 = np.log(df2['last_review'] + 1)

plt.figure(figsize=(10,2))
plt.subplot(1,2,1)
sns.histplot(X1, bins=10)
plt.subplot(1,2,2)
sns.histplot(X2, bins=10)
plt.show()

In [ ]:
df2.isna().sum()  # me dejo neighbourhood para más adelante

# Rescaling

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

df4 = df3.copy()

In [ ]:
scaler = MinMaxScaler()
scaler = RobustScaler()

for col in num_cols:
    X = df4[[col]]
    X_new = scaler.fit_transform(X)

    plt.figure(figsize=(10,2))
    plt.subplot(1,2,1)
    sns.histplot(X)
    plt.subplot(1,2,2)
    sns.histplot(X_new)
    plt.show()

# Cambio de distribuciones numéricas

In [ ]:
df3 = df2.copy()

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

kbd = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')

X = df2[['minimum_nights']]
X_new = kbd.fit_transform(X)

In [ ]:
X1 = df2[['minimum_nights']]
X2 = df3[['minimum_nights']]

plt.figure(figsize=(10,2))
plt.subplot(1,2,1)
sns.histplot(X)
plt.subplot(1,2,2)
sns.histplot(X_new)
plt.show()

In [ ]:
for col in num_cols:
    kbd = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')
    X = df2[[col]]
    X_new = kbd.fit_transform(X)

    plt.figure(figsize=(10,2))
    plt.subplot(1,2,1)
    sns.histplot(X)
    plt.subplot(1,2,2)
    sns.histplot(X_new)
    plt.show()

In [ ]:
from sklearn.preprocessing import QuantileTransformer

qt = QuantileTransformer(n_quantiles=100, output_distribution='normal', random_state=42)
X_new = qt.fit_transform(X)

In [ ]:
for col in num_cols:
    qt = QuantileTransformer(n_quantiles=100, output_distribution='normal', random_state=42)
    X = df2[[col]]
    X_new = qt.fit_transform(X)

    plt.figure(figsize=(10,2))
    plt.subplot(1,2,1)
    sns.histplot(X)
    plt.subplot(1,2,2)
    sns.histplot(X_new)
    plt.show()

In [ ]:
from sklearn.preprocessing import PowerTransformer

for col in num_cols:
    pt_yeo = PowerTransformer(method='yeo-johnson')
    
    X = df2[[col]]
    X_new = pt_yeo.fit_transform(X)

    plt.figure(figsize=(10,2))
    plt.subplot(1,2,1)
    sns.histplot(X)
    plt.subplot(1,2,2)
    sns.histplot(X_new)
    plt.show()

In [ ]:
for col in num_cols:
    if not col in ['latitude', 'longitude']:
        pt_yeo = PowerTransformer(method='box-cox')
        
        X = df2[[col]]
        X_pos = X + 1
        X_new = pt_yeo.fit_transform(X_pos)
    
        plt.figure(figsize=(10,2))
        plt.subplot(1,2,1)
        sns.histplot(X)
        plt.subplot(1,2,2)
        sns.histplot(X_new)
        plt.show()

In [ ]:
for col in num_cols:
    if not col in ['latitude', 'longitude']:
        X = df2[[col]]
        X_new = np.log(X + 1)
    
        plt.figure(figsize=(10,2))
        plt.subplot(1,2,1)
        sns.histplot(X)
        plt.subplot(1,2,2)
        sns.histplot(X_new)
        plt.show()

In [ ]:
# kbins-transform
for col in ['minimum_nights', 'number_of_reviews']:
    kbd = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')
    X = df2[[col]]
    X_new = kbd.fit_transform(X)
    
    df3[col] = X_new

# quantile-transform
for col in ['reviews_per_month', 'calculated_host_listings_count']:
    qt = QuantileTransformer(n_quantiles=100, output_distribution='normal', random_state=42)
    X = df2[[col]]
    X_new = qt.fit_transform(X)
    
    df3[col] = X_new

# power-transform
for col in ['availability_365', 'last_review']:
    qt = QuantileTransformer(n_quantiles=100, output_distribution='normal', random_state=42)
    X = df2[[col]]
    X_new = qt.fit_transform(X)
    
    df3[col] = X_new

# Transformaciones categóricas

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

In [ ]:
df5 = df4.copy()

In [ ]:
le_group = LabelEncoder()
le_neighbourhood = LabelEncoder()

df5['neighbourhood_group_encoded'] = le_group.fit_transform(df4[['neighbourhood_group']])
df5['neighbourhood_encoded'] = le_neighbourhood.fit_transform(df4[['neighbourhood']])
le_neighbourhood.classes_
le_group.classes_

In [ ]:
# df_na_neigh
# df4.iloc[147]
# df5.iloc[147]

In [ ]:
df5.loc[df_na_neigh.index, ['neighbourhood_group_encoded', 'neighbourhood_encoded']] = np.nan

In [ ]:
df5.isna().sum()

In [ ]:
df6 = df5.copy()

cols_for_imputation = [
    'latitude', 
    'longitude', 
    'neighbourhood_group_encoded', 
    'neighbourhood_encoded'
]

df_impute = df6[cols_for_imputation].copy()

imputer = KNNImputer(n_neighbors=5, weights='distance')
df6[cols_for_imputation] = imputer.fit_transform(df6[cols_for_imputation])

In [ ]:
df6.isna().sum()

In [ ]:
# df6.iloc[147]

In [ ]:
# inverse_transform

df6['neighbourhood_group_encoded'] = np.round(df6['neighbourhood_group_encoded']).astype(int)
df6['neighbourhood_encoded'] = np.round(df6['neighbourhood_encoded']).astype(int)

df6['neighbourhood_group'] = le_group.inverse_transform(df6['neighbourhood_group_encoded'])
df6['neighbourhood'] = le_neighbourhood.inverse_transform(df6['neighbourhood_encoded'])

In [ ]:
X1 = df6.iloc[df5.neighbourhood_group.isna()]['longitude']
X2 = df6.iloc[df5.neighbourhood_group.isna()]['latitude']
X3 = df6.iloc[df5.neighbourhood_group.isna()]["neighbourhood_group"]

sns.jointplot(x=X1, y=X2, hue=X3)

In [ ]:
df6.drop(columns=['neighbourhood_group_encoded','neighbourhood_encoded'], inplace=True)

## OneHotEncoder

In [ ]:
df7 = df6.copy()

In [ ]:
# neighbourhood_group
X = df7[["neighbourhood_group"]]
ohe_ng = OneHotEncoder()
X_new_ng = ohe_ng.fit_transform(X)

# room_type
X = df7[["room_type"]]
ohe_rt = OneHotEncoder(drop='first')
X_new_rt = ohe_rt.fit_transform(X)

In [ ]:
# neighbourhood
categories = [[
    'Williamsburg', 'Bedford-Stuyvesant', 'Harlem', 'Bushwick', 'Upper West Side',
    "Hell's Kitchen", 'East Village', 'Upper East Side', 'Crown Heights', 'Midtown'
]]

# sacamos la lista de aquí
# df_aux = df6.groupby('neighbourhood').size().reset_index()
# df_aux.columns = ['neighbourhood', 'tot']
# df_aux.sort_values('tot', ascending=False).head(10)['neighbourhood'].to_list()

X = df7[["neighbourhood"]]
ohe_n = OneHotEncoder(categories=categories, handle_unknown='ignore')
X_new_n = ohe_n.fit_transform(X)

In [ ]:
df8 = df7.copy()

df8 = df7.drop(columns=['neighbourhood_group', 'room_type', 'neighbourhood'])

# neighbourhood_group
df_ohe_ng = pd.DataFrame(
    X_new_ng.toarray(),
    columns=ohe_ng.get_feature_names_out(['neighbourhood_group']),
    index=df7.index
)

# room_type
df_ohe_rt = pd.DataFrame(
    X_new_rt.toarray(),
    columns=ohe_rt.get_feature_names_out(['room_type']),
    index=df7.index
)

# neighbourhood
df_ohe_n = pd.DataFrame(
    X_new_n.toarray(),
    columns=ohe_n.get_feature_names_out(['neighbourhood']),
    index=df7.index
)

# Merges por índice
df8 = df8.merge(df_ohe_ng, left_index=True, right_index=True)
df8 = df8.merge(df_ohe_rt, left_index=True, right_index=True)
df8 = df8.merge(df_ohe_n,  left_index=True, right_index=True)

In [ ]:
pd.set_option('display.max_columns', None)
df8.head()

# Model

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import *

X = df8.drop(columns='price')
# y = df8['price']
y = np.log(df8['price'])

model = LinearRegression()
model.fit(X, y)
y_hat = model.predict(X)

r2_score(y_true=y, y_pred=y_hat)